In [16]:
import asyncio
import os
import planet
from planet import order_request, reporting, Session
import geopandas as gpd
import shapely
import json

## Input

#### in this we will be using the shapefile that we created from task11_skysat_scene_filtering of the chosen scenes that are not overlapping

In [28]:
ids_path= 'data/no_overlapping_30_percent.shp'

out_path = 'data/skysat_planet_data'

item_type='SkySatCollect'

product_bundle='visual'

---
# CODE

In [21]:
ids=gpd.read_file(ids_path)['id'].values

## Utilities

In [22]:
async def order(session, ids, item_type, product_bundle, order_name):
    """This function creates the scene order using the Orders API, waits, and downloads the files
    into the current working directory."""

    # Initialize the Orders API client so we can clip and download scenes:
    client = session.client("orders")

    # Define the order request details for the Orders API:
    order_info = order_request.build_request(
        name=order_name,
        products=[
            order_request.product(
                item_ids=ids, product_bundle=product_bundle, item_type=item_type
            )
        ],
    )

    # Create the order, and use a progress bar:
    with reporting.StateBar(state="creating") as reporter:
        order = await client.create_order(order_info)
        reporter.update(state="created", order_id=order["id"])
        await client.wait(order["id"], progress_bar=True)

    # Download the order
    await client.download_order(order["id"], progress_bar=True)

## Download files

In [29]:
async def main(ids, item_type, product_bundle):
    # Initialize an API session to use in this request:
    async with Session() as sess:

        # Order and download the clipped scenes:
        await order(sess, ids, item_type, product_bundle, 'surge_planet_download')

#await main(ids, item_type, product_bundle)

00:00 - order  - state: creating


BadQuery: {"field":{"Details":[{"message":"Item ID s / Item Type SkySatCollect doesn't exist"}]},"general":[{"message":"Unable to accept order"}]}
